<a href="https://colab.research.google.com/github/Asmina-hub/LLM-from-scratch/blob/main/LLM_Input_output_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data sampling with a sliding window

In [ ]:
!pip install tiktoken
!pip install torch
!pip install PyPDF2

In [ ]:
import tiktoken
import torch
from PyPDF2 import PdfReader
import numpy as np
from torch.utils.data import Dataset,DataLoader

In [ ]:
with open('harrypotter.pdf','rb') as f:
  reader =PdfReader(f)
  pdf_text = ""
  for i in range(11,len(reader.pages)-1):
    page = reader.pages[i]
    pdf_text = pdf_text + page.extract_text()

  print(f"Total extracted characters: {len(pdf_text)}")
  print("First 500 characters of the extracted text:")
  print(pdf_text[:500])



Total extracted characters: 6279327
First 500 characters of the extracted text:
M
	
CHAPTER		ONE
THE	BOY	WHO	LIVED
r.	and	Mrs.	Dursley,	of	number	four,	Privet	Drive,	were	proud	to	say
that	they	were	perfectly	normal,	thank	you	very	much.	They	were	the
last	people	you’d	expect	to	be	involved	in	anything	strange	or	mysterious,
because	they	just	didn’t	hold	with	such	nonsense.
Mr.	Dursley	was	the	director	of	a	firm	called	Grunnings,	which	made	drills.
He	was	a	big,	beefy	man	with	hardly	any	neck,	although	he	did	have	a	very
large	mustache.	Mrs.	Dursley	was	thin	and	blonde	and	


In [ ]:
class InputOuptputDataset(Dataset):
    def __init__(self,txt,context_length,tokenizer,stride):
      self.txt=txt
      self.context_length=context_length
      self.tokenizer=tokenizer
      self.stride = stride

      self.input=[]
      self.output=[]

      encoded_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})
      for i in range(0,(len(encoded_ids)-context_length),stride):
        self.input.append(torch.tensor(encoded_ids[i:i+context_length]))
        self.output.append(torch.tensor(encoded_ids[i+1:i+context_length+1]))



    def __len__(self):
      return len(self.input)

    def __getitem__(self,idx):
      context=self.input[idx]
      target=self.output[idx]
      return context,target


In [ ]:
def create_dataloader(txt,batch_size,shuffle=True,max_length=256,stride=128,drop_last=True,num_workers=0):
  tokenizer=tiktoken.get_encoding('gpt2')
  dataset = InputOuptputDataset(txt,max_length,tokenizer,stride)
  dataloader_init = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
  return dataloader_init



In [ ]:
dataloader = create_dataloader(pdf_text,batch_size=1,shuffle=False,max_length=4,stride=1,drop_last=True,num_workers=0)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[ 44, 198, 197, 198]]), tensor([[  198,   197,   198, 41481]])]


In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[  198,   197,   198, 41481]]), tensor([[  197,   198, 41481,   197]])]
